# Experimento 3: Clustering Jerárquico

Este notebook implementa clustering jerárquico aglomerativo sobre la matriz normalizada de trayectorias académicas. Se evalúan diferentes configuraciones de número de clusters y método de enlace mediante métricas internas de validación.

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score

print("Entorno listo")

Entorno listo


In [2]:
ruta = Path("../data/processed/X_clustering.csv")

X = pd.read_csv(ruta)

X.head()

,periodos_registrados,promedio_global,creditos_totales,cursos_aprobados,cursos_desaprobados,cursos_previo_inicio,promedio_inicial,promedio_final,tendencia,tasa_aprobacion,variabilidad_nota
0,-0.487231,0.516412,-0.402032,-0.464573,-0.711080,-0.596566,0.147024,0.486179,0.433139,0.673603,-0.961629
1,-0.487231,-0.420977,-0.487768,0.028994,0.942910,1.321249,0.383717,-1.921609,-2.051139,-0.304771,1.363608
2,-0.487231,-0.366903,-0.487768,0.028994,0.777511,1.129467,0.265371,-1.921609,-2.009536,-0.222381,1.221827
3,-1.416016,-2.899670,-1.387992,-1.385899,0.446713,0.745904,-1.825423,-1.754733,-1.108143,-2.636562,1.369857
4,0.441553,0.189696,0.555349,0.950319,-0.545681,-0.596566,-0.629817,0.831851,1.050932,0.599907,-0.535305


In [3]:
resultados_jerarquico = []


for linkage in ["ward", "complete", "average"]:

    for k in range(2, 11):

        modelo = AgglomerativeClustering(
            n_clusters=k,
            linkage=linkage
        )


        etiquetas = modelo.fit_predict(X)


        silhouette = silhouette_score(
            X,
            etiquetas
        )


        davies = davies_bouldin_score(
            X,
            etiquetas
        )


        resultados_jerarquico.append({
            "linkage": linkage,
            "k": k,
            "silhouette": silhouette,
            "davies_bouldin": davies
        })


resultados_jerarquico = pd.DataFrame(
    resultados_jerarquico
)


resultados_jerarquico

,linkage,k,silhouette,davies_bouldin
0,ward,2,0.497297,0.815482
1,ward,3,0.383058,0.972675
2,ward,4,0.417991,1.125397
3,ward,5,0.421194,0.940267
4,ward,6,0.430018,0.919620
5,ward,7,0.407951,0.957241
6,ward,8,0.339847,1.037705
7,ward,9,0.355756,0.949607
8,ward,10,0.337022,0.969030
9,complete,2,0.439305,1.023802


In [4]:
min_grupo = int(np.ceil(len(X) * 0.05))

revision = []

for _, fila in resultados_jerarquico.iterrows():
    modelo = AgglomerativeClustering(
        n_clusters=int(fila["k"]),
        linkage=fila["linkage"]
    )

    etiquetas = modelo.fit_predict(X)
    tamanos = pd.Series(etiquetas).value_counts()

    revision.append({
        "linkage": fila["linkage"],
        "k": int(fila["k"]),
        "silhouette": fila["silhouette"],
        "davies_bouldin": fila["davies_bouldin"],
        "grupo_minimo": int(tamanos.min()),
        "grupo_maximo": int(tamanos.max()),
        "cumple": tamanos.min() >= min_grupo
    })

revision = pd.DataFrame(revision)

validos = revision[revision["cumple"]].sort_values(
    ["silhouette", "davies_bouldin"],
    ascending=[False, True]
)

print("Tamaño mínimo propuesto:", min_grupo)
print("\nCONFIGURACIONES QUE CUMPLEN:")
print(validos.to_string(index=False))


Tamaño mínimo propuesto: 13

CONFIGURACIONES QUE CUMPLEN:
 linkage  k  silhouette  davies_bouldin  grupo_minimo  grupo_maximo  cumple
    ward  2    0.497297        0.815482            55           189    True
complete  2    0.439305        1.023802            37           207    True
    ward  4    0.417991        1.125397            26            86    True
    ward  3    0.383058        0.972675            55           112    True


In [5]:

if validos.empty:
    raise ValueError(
        "Ninguna configuración cumple el tamaño mínimo."
    )

mejor_config = validos.iloc[0]

print("CONFIGURACIÓN SELECCIONADA:")
print(mejor_config)

CONFIGURACIÓN SELECCIONADA:
linkage               ward
k                        2
silhouette        0.497297
davies_bouldin    0.815482
grupo_minimo            55
grupo_maximo           189
cumple                True
Name: 0, dtype: object


In [6]:
modelo_final = AgglomerativeClustering(
    n_clusters=int(mejor_config["k"]),
    linkage=mejor_config["linkage"]
)


clusters = modelo_final.fit_predict(X)

clusters

array([0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0,
       0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0,
       0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0,
       1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0,
       1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0,
       1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0,
       0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0,
       1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1,
       0, 0])

In [7]:
pd.Series(clusters).value_counts()

0    189
1     55
Name: count, dtype: int64

In [8]:
silhouette_final = silhouette_score(
    X,
    clusters
)

davies_final = davies_bouldin_score(
    X,
    clusters
)


print("Silhouette:", silhouette_final)
print("Davies-Bouldin:", davies_final)

Silhouette: 0.49729661877713843
Davies-Bouldin: 0.8154819311369225
 0.49729661877713843
Davies-Bouldin: 0.8154819311369225


In [9]:
resultado_jerarquico = pd.DataFrame({
    "modelo": ["Jerárquico"],
    "configuracion": [
        f'{mejor_config["linkage"]}, k={int(mejor_config["k"])}'
    ],
    "silhouette": [silhouette_final],
    "davies_bouldin": [davies_final]
})


resultado_jerarquico.to_csv(
    "../results/jerarquico/evaluacion_jerarquico.csv",
    index=False,
    encoding="utf-8-sig"
)


resultado_jerarquico

,modelo,configuracion,silhouette,davies_bouldin
0,Jerárquico,"ward, k=2",0.497297,0.815482


In [10]:
df_original = pd.read_csv(
    "../data/processed/trayectorias_academicas.csv"
)

resultado_clusters = df_original.copy()

resultado_clusters["cluster"] = clusters

resultado_clusters.groupby(
    "cluster"
).mean(numeric_only=True)

,id_estudiante,periodos_registrados,promedio_global,creditos_totales,cursos_aprobados,cursos_desaprobados,cursos_previo_inicio,promedio_inicial,promedio_final,tendencia,tasa_aprobacion,variabilidad_nota
cluster,,,,,,,,,,,,
0,6.218046e+07,7.412698,14.297740,163.359788,54.275132,2.206349,1.185185,14.144897,14.476311,0.331414,0.958375,2.052679
1,6.114520e+07,3.690909,9.386255,82.509091,22.527273,11.490909,9.727273,13.603030,1.339594,-12.263437,0.616419,6.099285


In [11]:

caracterizacion_jerarquico = (
    resultado_clusters
    .drop(columns=["id_estudiante"], errors="ignore")
    .groupby("cluster")
    .mean(numeric_only=True)
)

caracterizacion_jerarquico.to_csv(
    "../results/jerarquico/caracterizacion_clusters.csv",
    encoding="utf-8-sig"
)


evaluacion_guardada = pd.read_csv(
    "../results/jerarquico/evaluacion_jerarquico.csv"
)

print("CONFIGURACIÓN GUARDADA:")
print(evaluacion_guardada.to_string(index=False))

print("\nESTUDIANTES POR CLÚSTER:")
print(pd.Series(clusters).value_counts().sort_index())

print("\nCARACTERIZACIÓN:")
print(caracterizacion_jerarquico)


CONFIGURACIÓN GUARDADA:
    modelo configuracion  silhouette  davies_bouldin
Jerárquico     ward, k=2    0.497297        0.815482

ESTUDIANTES POR CLÚSTER:
0    189
1     55
Name: count, dtype: int64

CARACTERIZACIÓN:
         periodos_registrados  promedio_global  creditos_totales  \
cluster                                                            
0                    7.412698        14.297740        163.359788   
1                    3.690909         9.386255         82.509091   

         cursos_aprobados  cursos_desaprobados  cursos_previo_inicio  \
cluster                                                                
0               54.275132             2.206349              1.185185   
1               22.527273            11.490909              9.727273   

         promedio_inicial  promedio_final  tendencia  tasa_aprobacion  \
cluster                                                                 
0               14.144897       14.476311   0.331414         0.958375   
1

In [12]:

from pathlib import Path

# Guardar las asignaciones del modelo seleccionado
asignaciones_jerarquico = pd.DataFrame({
    "fila": range(len(clusters)),
    "cluster": clusters
})

# Verificar la cantidad de estudiantes
assert len(asignaciones_jerarquico) == len(X)
assert mejor_config["linkage"] == "ward"
assert int(mejor_config["k"]) == 2

# Crear carpeta si no existe
ruta = Path("../results/jerarquico")
ruta.mkdir(parents=True, exist_ok=True)

# Guardar las asignaciones
asignaciones_jerarquico.to_csv(
    ruta / "asignaciones_jerarquico.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Asignaciones jerárquicas guardadas correctamente")
print("Total de estudiantes:", len(asignaciones_jerarquico))

print("\nDistribución:")
print(
    asignaciones_jerarquico["cluster"]
    .value_counts()
    .sort_index()
)


Asignaciones jerárquicas guardadas correctamente
Total de estudiantes: 244

Distribución:
cluster
0    189
1     55
Name: count, dtype: int64
